<a href="https://colab.research.google.com/github/CamiloVga/IA-Codes/blob/main/DeepResearch_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Deep Research Agent - Ultra Minimalista
!pip install langchain langchain-openai langchain-anthropic tavily-python

import os
from google.colab import userdata
from tavily import TavilyClient
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

# ============================================================================
# CONFIGURACIÓN PRINCIPAL
# ============================================================================

# Secretos
TAVILY_KEY = userdata.get('TAVILY_KEY')
OPENAI_KEY = userdata.get('OPENAI_API_KEY')
ANTHROPIC_KEY = userdata.get('ANTHROPIC_API_KEY')

# ============================================================================
# CONFIGURACIÓN DE IA - CAMBIAR AQUÍ
# ============================================================================

# Elegir provider: "openai" o "anthropic"
AI_PROVIDER = "openai"

# Modelos disponibles:
# OpenAI (2025):
#   - "gpt-4.1"             → razonador más reciente, recomendado
#   - "gpt-4.1-mini"        → versión ligera de gpt-4.1 (eficiente)
#   - "gpt-4o"              → razonador multimodal, rápido y económico
# Anthropic (2025):
#   - "claude-opus-4-20250514"     → modelo más avanzado (Claude 4)
#   - "claude-sonnet-4-20250514"   → balance entre costo y calidad
#   - "claude-3-opus-20240229"     → aún válido para razonamiento extenso

AI_MODEL = "gpt-4.1"  # Puedes cambiar aquí según tu necesidad

# Hiperparámetros
TEMPERATURE = 0.1    # 0.0 = determinista, 1.0 = creativo
MAX_TOKENS = 2000    # Longitud máxima respuesta

# ============================================================================
# CONFIGURACIÓN DE TAVILY - CAMBIAR AQUÍ
# ============================================================================

# Configuración básica (activa)
SEARCH_MAX_RESULTS = 5
SEARCH_TOPIC = "general"  # "general", "news", "finance"

# Configuraciones avanzadas (comentadas - desactivadas)
# SEARCH_TIME_RANGE = "day"  # "day", "week", "month", "year"
# SEARCH_INCLUDE_ANSWER = "advanced"  # "basic", "advanced"
# SEARCH_INCLUDE_RAW_CONTENT = True  # True, False
# SEARCH_INCLUDE_DOMAINS = ["https://www.elespectador.com/"]
# SEARCH_EXCLUDE_DOMAINS = ["https://www.eltiempo.com/"]
# SEARCH_LOCATION = "CO"  # Código país
# SEARCH_LANGUAGE = "es"  # Idioma

# ============================================================================
# INICIALIZACIÓN
# ============================================================================

def init_ai():
    """Inicializa el modelo de IA según configuración"""
    if AI_PROVIDER == "openai":
        return ChatOpenAI(
            model=AI_MODEL,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            api_key=OPENAI_KEY
        )
    elif AI_PROVIDER == "anthropic":
        return ChatAnthropic(
            model=AI_MODEL,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            api_key=ANTHROPIC_KEY
        )

def init_tavily():
    """Inicializa cliente Tavily"""
    return TavilyClient(api_key=TAVILY_KEY)

# Inicializar
llm = init_ai()
tavily = init_tavily()

# ============================================================================
# FUNCIONES CORE
# ============================================================================

def search_web(query):
    """Busca en web con Tavily"""
    try:
        # Configuración básica
        search_params = {
            "query": query,
            "max_results": SEARCH_MAX_RESULTS,
            "topic": SEARCH_TOPIC
        }

        # Configuraciones avanzadas (descomentar para activar)
        # search_params.update({
        #     "time_range": SEARCH_TIME_RANGE,
        #     "include_answer": SEARCH_INCLUDE_ANSWER,
        #     "include_raw_content": SEARCH_INCLUDE_RAW_CONTENT,
        #     "include_domains": SEARCH_INCLUDE_DOMAINS,
        #     "exclude_domains": SEARCH_EXCLUDE_DOMAINS,
        #     "location": SEARCH_LOCATION,
        #     "language": SEARCH_LANGUAGE
        # })

        response = tavily.search(**search_params)

        return [
            {
                "title": r.get("title", ""),
                "url": r.get("url", ""),
                "content": r.get("content", "")
            }
            for r in response.get("results", [])
        ]
    except Exception as e:
        print(f"Error búsqueda: {e}")
        return []

def generate_queries(topic):
    """Genera consultas de investigación"""
    return [
        f"{topic} overview",
        f"{topic} recent trends",
        f"{topic} applications",
        f"{topic} challenges"
    ]

def research(topic):
    """Investiga un tema"""
    print(f"🔍 Investigando: {topic}")

    # Generar consultas
    queries = generate_queries(topic)

    # Buscar información
    all_results = []
    for query in queries:
        results = search_web(query)
        all_results.extend(results[:2])  # Top 2 por consulta

    # Preparar contexto
    context = "\n\n".join([
        f"FUENTE: {r['title']}\nURL: {r['url']}\nCONTENIDO: {r['content'][:400]}..."
        for r in all_results[:6]  # Top 6 total
    ])

    # Generar reporte
    prompt = f"""
    Crea un reporte sobre "{topic}" usando la información proporcionada.

    Estructura:
    - Introducción
    - Puntos clave
    - Tendencias actuales
    - Aplicaciones
    - Conclusiones

    Formato markdown. Cita fuentes.

    INFORMACIÓN:
    {context}
    """

    print("📝 Generando reporte...")
    response = llm.invoke(prompt)
    return response.content

# ============================================================================
# FUNCIÓN PRINCIPAL
# ============================================================================

def investigar(topic):
    """Función principal para investigar cualquier tema"""
    return research(topic)

# ============================================================================
# PRUEBAS Y EJEMPLOS
# ============================================================================

def test_agent():
    """Prueba el agente con tema de ejemplo"""

    print("=== CONFIGURACIÓN ACTUAL ===")
    print(f"Provider: {AI_PROVIDER}")
    print(f"Modelo: {AI_MODEL}")
    print(f"Temperature: {TEMPERATURE}")
    print(f"Max tokens: {MAX_TOKENS}")
    print(f"Tavily max results: {SEARCH_MAX_RESULTS}")
    print(f"Tavily topic: {SEARCH_TOPIC}")
    print()

    # Tema de prueba
    test_topic = "Inteligencia Artificial Generativa"

    print(f"🧪 PROBANDO CON: {test_topic}")
    print("="*50)

    try:
        reporte = investigar(test_topic)
        print("✅ ÉXITO!")
        print(f"Longitud: {len(reporte)} caracteres")
        print("\nPREVIEW:")
        print(reporte)
        return reporte
    except Exception as e:
        print(f"❌ ERROR: {e}")
        return None
# Prueba de funcionamiento
test_agent()

# ============================================================================
# FUNCIÓN DE INFERENCIA RÁPIDA
# ============================================================================

# Ejemplo básico
reporte = investigar("Dame las noticias de hoy")
print(reporte)



=== CONFIGURACIÓN ACTUAL ===
Provider: openai
Modelo: gpt-4.1
Temperature: 0.1
Max tokens: 2000
Tavily max results: 5
Tavily topic: general

🧪 PROBANDO CON: Inteligencia Artificial Generativa
🔍 Investigando: Inteligencia Artificial Generativa
📝 Generando reporte...
✅ ÉXITO!
Longitud: 4393 caracteres

PREVIEW:
# Reporte sobre Inteligencia Artificial Generativa

## Introducción

La **Inteligencia Artificial Generativa** (GenAI, por sus siglas en inglés) es una rama de la inteligencia artificial que utiliza modelos generativos para crear nuevos contenidos, como texto, imágenes, audio, video y otros tipos de datos. Esta tecnología ha experimentado un crecimiento exponencial en los últimos años, redefiniendo los límites de lo que las máquinas pueden lograr y abriendo nuevas posibilidades en diversos sectores industriales y creativos [Wikipedia](https://en.wikipedia.org/wiki/Generative_artificial_intelligence).

---

## Puntos clave

- **Modelos generativos:** Utiliza modelos como autoencode